# 13 · Feature engineering de los 17 partidos (experimento de ampliación)

Replica la lógica del notebook 03 (sin modificarlo) para los 5 core + los 12 partidos nuevos y genera una **tabla larga** en `data/processed/dataset_modelado_ampliacion.csv`.

- Mismas ventanas, mismas features climáticas, de NDVI y v2 (VPD + anomalía de NDVI) que el 03.
- Normal de NDVI **por partido**, campañas 2000-2015 (igual que el 03), guardada en un archivo aparte (`ndvi_normal_ampliacion_2000_2015.csv`) para **no pisar** `ndvi_normal_2000_2015.csv`.
- Metadatos (no son features): `grupo` (core / ampliacion), `zona` (core / nucleo / transicion / oeste) y `grupo_celda` (celda de la grilla NASA POWER; los partidos que comparten celda tienen el mismo valor).
- Chequeos: 425 filas (25 campañas × 17 partidos) y que las filas de los 5 core sean idénticas a `dataset_modelado_soja.csv`.

In [1]:
# Paths del proyecto: centralizados en src/config.py
import sys
from pathlib import Path

_raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if _raiz is None:
    raise FileNotFoundError("No se encontró src/config.py: abrí el notebook desde la carpeta del proyecto")
if str(_raiz / "src") not in sys.path:
    sys.path.insert(0, str(_raiz / "src"))
from config import (PROJECT_ROOT, DATA_RAW, DATA_PROCESSED, OUTPUTS,
                    PARTIDOS, PARTIDOS_AMPLIACION, DATA_RAW_AMPLIACION)

## 1. Partidos, grupos y zonas

In [2]:
import unicodedata
import pandas as pd
import numpy as np

def nfc(s):
    """Normaliza Unicode (NFC) y espacios: clave estable para cruzar fuentes con tildes."""
    return unicodedata.normalize("NFC", str(s)).strip()

PARTIDOS_CORE = [nfc(p) for p in PARTIDOS]                 # los 5 core (orden del 03)
PARTIDOS_NUEVOS = [nfc(p) for p in PARTIDOS_AMPLIACION]    # los 12 nuevos
PARTIDOS_17 = PARTIDOS_CORE + PARTIDOS_NUEVOS
assert len(PARTIDOS_17) == 17 and len(set(PARTIDOS_17)) == 17

GRUPO = {p: "core" for p in PARTIDOS_CORE} | {p: "ampliacion" for p in PARTIDOS_NUEVOS}

# Zona: SOLO metadato, no entra como feature
ZONA = {p: "core" for p in PARTIDOS_CORE}
ZONA |= {nfc(p): "nucleo" for p in ["Pergamino", "Colón", "Rojas", "Salto"]}
ZONA |= {nfc(p): "transicion" for p in ["Chacabuco", "Bragado", "9 de Julio", "General Viamonte"]}
ZONA |= {nfc(p): "oeste" for p in ["General Villegas", "Pehuajó", "Carlos Tejedor", "Trenque Lauquen"]}
assert set(ZONA) == set(PARTIDOS_17), set(PARTIDOS_17) ^ set(ZONA)

print(pd.Series(ZONA).value_counts().to_dict())

{'core': 5, 'nucleo': 4, 'transicion': 4, 'oeste': 4}


## 2. Carga de datos (core + ampliación)

In [3]:
df_rinde = pd.read_csv(DATA_RAW / "magyp" / "soja-serie-1941-2024.csv")

df_clima = pd.concat([
    pd.read_csv(DATA_RAW / "nasa_power" / "nasa_power_zona_coop_1999_2025.csv", parse_dates=["fecha"]),
    pd.read_csv(DATA_RAW_AMPLIACION / "nasa_power" / "nasa_power_ampliacion_1999_2025.csv", parse_dates=["fecha"]),
], ignore_index=True)

df_ndvi = pd.concat([
    pd.read_csv(DATA_RAW / "ndvi" / "ndvi_modis_zona_coop_2000_2025.csv", parse_dates=["fecha"]),
    pd.read_csv(DATA_RAW_AMPLIACION / "ndvi" / "ndvi_modis_ampliacion_2000_2025.csv", parse_dates=["fecha"]),
], ignore_index=True)

# Clave de cruce normalizada
df_rinde["departamento"] = df_rinde["departamento"].map(nfc)
df_clima["partido"] = df_clima["partido"].map(nfc)
df_ndvi["partido"] = df_ndvi["partido"].map(nfc)

# Cada fuente debe tener exactamente los 17 partidos, con el mismo nombre
assert set(df_clima["partido"]) == set(PARTIDOS_17), set(df_clima["partido"]) ^ set(PARTIDOS_17)
assert set(df_ndvi["partido"]) == set(PARTIDOS_17), set(df_ndvi["partido"]) ^ set(PARTIDOS_17)
faltan_magyp = set(PARTIDOS_17) - set(df_rinde.loc[df_rinde["provincia"] == "Buenos Aires", "departamento"])
assert not faltan_magyp, f"Partidos sin match en MAGyP: {faltan_magyp}"

print(f"Rinde: {df_rinde.shape}")
print(f"Clima: {df_clima.shape}  ({df_clima['partido'].nunique()} partidos)")
print(f"NDVI:  {df_ndvi.shape}  ({df_ndvi['partido'].nunique()} partidos)")

Rinde: (12602, 11)
Clima: (159358, 11)  (17 partidos)
NDVI:  (9860, 5)  (17 partidos)


## 3. Celda de la grilla NASA POWER (`grupo_celda`)

Se calcula con el punto (lat, lon) que realmente se usó para descargar el clima de cada partido (manual en los core, centroide GAUL en los nuevos). Grilla: 0,5° de latitud × 0,625° de longitud.

In [4]:
def celda_nasa(lat, lon):
    clat = round(lat / 0.5) * 0.5
    clon = -180 + round((lon + 180) / 0.625) * 0.625
    return f"{clat:.3f}_{clon:.3f}"

puntos = df_clima.groupby("partido")[["lat", "lon"]].first()
assert (df_clima.groupby("partido")[["lat", "lon"]].nunique() == 1).all().all()   # un solo punto por partido
GRUPO_CELDA = {p: celda_nasa(r.lat, r.lon) for p, r in puntos.iterrows()}

celdas = pd.Series(GRUPO_CELDA).rename("grupo_celda").to_frame()
celdas["partidos_en_celda"] = celdas.groupby("grupo_celda")["grupo_celda"].transform("size")
print(f"Celdas distintas: {celdas['grupo_celda'].nunique()} para {len(celdas)} partidos")
for celda, g in celdas.groupby("grupo_celda"):
    if len(g) > 1:
        print(f"  {celda}: {', '.join(g.index)}")

Celdas distintas: 13 para 17 partidos
  -34.000_-60.625: Pergamino, Rojas
  -34.500_-61.250: General Arenales, Junín, Leandro N. Alem
  -35.000_-61.250: General Viamonte, Lincoln


## 4. Target (MAGyP)

In [5]:
target = df_rinde[
    (df_rinde["provincia"] == "Buenos Aires") &
    (df_rinde["departamento"].isin(PARTIDOS_17)) &
    (df_rinde["anio"] >= 2000)
].copy()

target["rinde_qq_ha"] = target["rendimiento_kgxha"] / 100
target = target[["anio", "departamento", "rinde_qq_ha", "superficie_sembrada_ha", "superficie_cosechada_ha"]].copy()
target.rename(columns={"anio": "campana", "departamento": "partido"}, inplace=True)

print(f"Target: {len(target)} filas")
print(f"Campañas: {target['campana'].min()} - {target['campana'].max()}")
print(target.groupby("partido").size().to_string())
assert target.groupby("partido").size().eq(25).all(), "Cada partido debe tener 25 campañas (2000-2024)"

Target: 425 filas
Campañas: 2000 - 2024
partido
9 de Julio          25
Bragado             25
Carlos Tejedor      25
Chacabuco           25
Colón               25
General Arenales    25
General Pinto       25
General Viamonte    25
General Villegas    25
Junín               25
Leandro N. Alem     25
Lincoln             25
Pehuajó             25
Pergamino           25
Rojas               25
Salto               25
Trenque Lauquen     25


## 5. Ventanas y features (misma lógica que el notebook 03)

In [6]:
# Campaña X = siembra Oct del año X, cosecha Mar-May del año X+1
VENTANAS = {
    "oct_nov": (10, 11),   # Solo siembra
    "oct_dic": (10, 12),   # Temprana
    "oct_ene": (10, 13),   # 13 = enero año siguiente
    "oct_feb": (10, 14),   # 14 = febrero año siguiente (período crítico completo)
    "oct_mar": (10, 15),   # 15 = marzo año siguiente
}

def mes_campana(fecha, campana):
    """Convierte fecha real a mes relativo de la campaña.
    Oct del año de siembra = 10, Nov = 11, Dic = 12,
    Ene del año siguiente = 13, Feb = 14, Mar = 15, etc."""
    if fecha.year == campana:
        return fecha.month
    elif fecha.year == campana + 1:
        return fecha.month + 12
    return None

PARTIDOS_FE = PARTIDOS_17   # única diferencia con el 03: 17 partidos en vez de 5

In [7]:
print("Calculando features climáticas...")
clima_features = []

for campana in range(2000, 2025):
    inicio = pd.Timestamp(f"{campana}-09-01")
    fin = pd.Timestamp(f"{campana+1}-05-31")

    dc = df_clima[(df_clima["fecha"] >= inicio) & (df_clima["fecha"] <= fin)].copy()
    dc["mes_camp"] = dc["fecha"].apply(lambda f: mes_campana(f, campana))

    for partido in PARTIDOS_FE:
        dp = dc[dc["partido"] == partido]
        if len(dp) == 0:
            continue

        row = {"campana": campana, "partido": partido}

        for ventana_nombre, (mes_ini, mes_fin) in VENTANAS.items():
            dv = dp[(dp["mes_camp"] >= mes_ini) & (dp["mes_camp"] <= mes_fin)]
            if len(dv) == 0:
                continue

            row[f"precip_acum_{ventana_nombre}"] = dv["PRECTOTCORR"].sum()
            row[f"temp_media_{ventana_nombre}"] = dv["T2M"].mean()
            row[f"temp_max_media_{ventana_nombre}"] = dv["T2M_MAX"].mean()
            row[f"temp_min_media_{ventana_nombre}"] = dv["T2M_MIN"].mean()
            row[f"rad_media_{ventana_nombre}"] = dv["ALLSKY_SFC_SW_DWN"].mean()
            row[f"humedad_media_{ventana_nombre}"] = dv["RH2M"].mean()
            row[f"dias_calor_extremo_{ventana_nombre}"] = (dv["T2M_MAX"] > 35).sum()
            row[f"dias_sin_lluvia_{ventana_nombre}"] = (dv["PRECTOTCORR"] < 1).sum()

        clima_features.append(row)

df_feat_clima = pd.DataFrame(clima_features)
print(f"Shape: {df_feat_clima.shape}")

Calculando features climáticas...
Shape: (425, 42)


In [8]:
print("Calculando features de NDVI...")
ndvi_features = []

for campana in range(2000, 2025):
    inicio = pd.Timestamp(f"{campana}-09-01")
    fin = pd.Timestamp(f"{campana+1}-05-31")

    dn = df_ndvi[(df_ndvi["fecha"] >= inicio) & (df_ndvi["fecha"] <= fin)].copy()
    dn["mes_camp"] = dn["fecha"].apply(lambda f: mes_campana(f, campana))

    for partido in PARTIDOS_FE:
        dp = dn[dn["partido"] == partido]
        if len(dp) == 0:
            continue

        row = {"campana": campana, "partido": partido}

        for ventana_nombre, (mes_ini, mes_fin) in VENTANAS.items():
            dv = dp[(dp["mes_camp"] >= mes_ini) & (dp["mes_camp"] <= mes_fin)]
            if len(dv) == 0:
                continue

            row[f"ndvi_mean_{ventana_nombre}"] = dv["ndvi"].mean()
            row[f"ndvi_max_{ventana_nombre}"] = dv["ndvi"].max()
            row[f"ndvi_min_{ventana_nombre}"] = dv["ndvi"].min()
            row[f"ndvi_std_{ventana_nombre}"] = dv["ndvi"].std()

            if len(dv) >= 2:
                x = np.arange(len(dv))
                slope = np.polyfit(x, dv["ndvi"].values, 1)[0]
                row[f"ndvi_slope_{ventana_nombre}"] = slope

        ndvi_features.append(row)

df_feat_ndvi = pd.DataFrame(ndvi_features)
print(f"Shape: {df_feat_ndvi.shape}")

Calculando features de NDVI...
Shape: (425, 27)


In [9]:
FIN_PERIODO_NORMAL = 2015   # "normal" de NDVI = campañas 2000-2015

def presion_vapor(t):
    """Presión de vapor de saturación (kPa), FAO-56 ec. 11."""
    return 0.6108 * np.exp(17.27 * t / (t + 237.3))

# VPD diario: (es(Tmax) + es(Tmin)) / 2 - ea(Tdew)
df_clima["vpd"] = (presion_vapor(df_clima["T2M_MAX"]) + presion_vapor(df_clima["T2M_MIN"])) / 2 - presion_vapor(df_clima["T2MDEW"])

# NDVI normal por partido y fecha del año (composiciones MODIS de 16 días: mismo día del año cada año)
dn_all = df_ndvi.copy()
dn_all["doy"] = dn_all["fecha"].dt.dayofyear
dn_all["campana_ref"] = np.where(dn_all["fecha"].dt.month >= 9, dn_all["fecha"].dt.year, dn_all["fecha"].dt.year - 1)
ndvi_normal = (dn_all[dn_all["campana_ref"] <= FIN_PERIODO_NORMAL]
               .groupby(["partido", "doy"])["ndvi"].mean().rename("ndvi_normal").reset_index())

# Archivo APARTE: no se pisa data/processed/ndvi_normal_2000_2015.csv (el de los 5 core)
OUT_NORMAL = DATA_PROCESSED / "ndvi_normal_ampliacion_2000_2015.csv"
ndvi_normal.to_csv(OUT_NORMAL, index=False)
dn_all = dn_all.merge(ndvi_normal, on=["partido", "doy"], how="left")
dn_all["ndvi_anom"] = dn_all["ndvi"] - dn_all["ndvi_normal"]

v2_features = []
for campana in range(2000, 2025):
    inicio, fin = pd.Timestamp(f"{campana}-09-01"), pd.Timestamp(f"{campana+1}-05-31")
    dc = df_clima[(df_clima["fecha"] >= inicio) & (df_clima["fecha"] <= fin)].copy()
    dc["mes_camp"] = dc["fecha"].apply(lambda f: mes_campana(f, campana))
    dn = dn_all[(dn_all["fecha"] >= inicio) & (dn_all["fecha"] <= fin)].copy()
    dn["mes_camp"] = dn["fecha"].apply(lambda f: mes_campana(f, campana))

    for partido in PARTIDOS_FE:
        dcp = dc[dc["partido"] == partido]
        dnp = dn[dn["partido"] == partido].sort_values("fecha")
        if len(dcp) == 0:
            continue
        row = {"campana": campana, "partido": partido}
        for ventana_nombre, (mes_ini, mes_fin) in VENTANAS.items():
            dv = dcp[(dcp["mes_camp"] >= mes_ini) & (dcp["mes_camp"] <= mes_fin)]
            if len(dv):
                row[f"vpd_media_{ventana_nombre}"] = dv["vpd"].mean()
            nv = dnp[(dnp["mes_camp"] >= mes_ini) & (dnp["mes_camp"] <= mes_fin)].dropna(subset=["ndvi_anom"])
            if len(nv):
                row[f"ndvi_anom_media_{ventana_nombre}"] = nv["ndvi_anom"].mean()
                row[f"ndvi_anom_ultimo_{ventana_nombre}"] = nv["ndvi_anom"].iloc[-1]   # estado a la fecha de corte
        v2_features.append(row)

# Orden de columnas: VPD (todas las ventanas), anomalía media, anomalía última
orden_v2 = [f"{p}_{v}" for p in ["vpd_media", "ndvi_anom_media", "ndvi_anom_ultimo"] for v in VENTANAS]
df_feat_v2 = pd.DataFrame(v2_features)[["campana", "partido"] + orden_v2]
print(f"Variables v2: {len(orden_v2)}  |  Shape: {df_feat_v2.shape}")
print(f"NDVI normal guardado en: {OUT_NORMAL}")

Variables v2: 15  |  Shape: (425, 17)


## 6. Dataset final (tabla larga, 17 partidos)

In [10]:
df_model = target.merge(df_feat_clima, on=["campana", "partido"], how="inner")
df_model = df_model.merge(df_feat_ndvi, on=["campana", "partido"], how="inner")
df_model = df_model.merge(df_feat_v2, on=["campana", "partido"], how="inner")   # v2: VPD + anomalía NDVI

# Metadatos (NO son features): justo después de 'partido'
df_model.insert(2, "grupo", df_model["partido"].map(GRUPO))
df_model.insert(3, "zona", df_model["partido"].map(ZONA))
df_model.insert(4, "grupo_celda", df_model["partido"].map(GRUPO_CELDA))
df_model = df_model.sort_values(["campana", "partido"]).reset_index(drop=True)

# --- Chequeos ---
assert len(df_model) == 25 * 17 == 425, f"Se esperaban 425 filas y hay {len(df_model)}"
assert df_model.groupby("partido").size().eq(25).all()
assert df_model[["grupo", "zona", "grupo_celda"]].notna().all().all()

META = ["campana", "partido", "grupo", "zona", "grupo_celda",
        "rinde_qq_ha", "superficie_sembrada_ha", "superficie_cosechada_ha"]
print(f"Shape final: {df_model.shape}")
print(f"Campañas: {df_model['campana'].min()} - {df_model['campana'].max()}")
print(f"Features: {df_model.shape[1] - len(META)} (sin contar target y metadatos)")
print(f"Nulls totales: {df_model.isnull().sum().sum()}")
print(df_model.groupby(["grupo", "zona"])["partido"].nunique())

Shape final: (425, 88)
Campañas: 2000 - 2024
Features: 80 (sin contar target y metadatos)
Nulls totales: 0
grupo       zona      
ampliacion  nucleo        4
            oeste         4
            transicion    4
core        core          5
Name: partido, dtype: int64


## 7. Chequeo: las filas de los 5 core deben ser idénticas a `dataset_modelado_soja.csv`

In [11]:
df_orig = pd.read_csv(DATA_PROCESSED / "dataset_modelado_soja.csv")
df_orig["partido"] = df_orig["partido"].map(nfc)

core_nuevo = df_model[df_model["grupo"] == "core"].drop(columns=["grupo", "zona", "grupo_celda"])
core_nuevo = core_nuevo.sort_values(["campana", "partido"]).reset_index(drop=True)
df_orig = df_orig.sort_values(["campana", "partido"]).reset_index(drop=True)

assert len(core_nuevo) == len(df_orig) == 125, (len(core_nuevo), len(df_orig))
assert set(core_nuevo.columns) == set(df_orig.columns), set(core_nuevo.columns) ^ set(df_orig.columns)
assert (core_nuevo[["campana", "partido"]].values == df_orig[["campana", "partido"]].values).all()

cols_num = [c for c in df_orig.columns if c not in ["campana", "partido"]]
max_dif = (core_nuevo[cols_num].astype(float) - df_orig[cols_num].astype(float)).abs().max()
print(f"Máxima diferencia absoluta en columnas de los core: {max_dif.max():.3e}")
no_ok = max_dif[max_dif > 1e-9]
assert no_ok.empty, f"Columnas que difieren del dataset original:\n{no_ok}"
print("OK: las 125 filas de los 5 core son idénticas a dataset_modelado_soja.csv")

Máxima diferencia absoluta en columnas de los core: 1.137e-13
OK: las 125 filas de los 5 core son idénticas a dataset_modelado_soja.csv


## 8. Guardar

In [12]:
output = DATA_PROCESSED / "dataset_modelado_ampliacion.csv"
assert output.name != "dataset_modelado_soja.csv"
output.parent.mkdir(parents=True, exist_ok=True)
df_model.to_csv(output, index=False)
print(f"Guardado en: {output}")
df_model.head()

,campana,partido,grupo,zona,grupo_celda,rinde_qq_ha,superficie_sembrada_ha,superficie_cosechada_ha,precip_acum_oct_nov,temp_media_oct_nov,...,ndvi_anom_media_oct_nov,ndvi_anom_media_oct_dic,ndvi_anom_media_oct_ene,ndvi_anom_media_oct_feb,ndvi_anom_media_oct_mar,ndvi_anom_ultimo_oct_nov,ndvi_anom_ultimo_oct_dic,ndvi_anom_ultimo_oct_ene,ndvi_anom_ultimo_oct_feb,ndvi_anom_ultimo_oct_mar
0,2000,9 de Julio,ampliacion,transicion,-35.500_-61.250,23.10,80000,62240,349.57,15.909508,...,0.073271,0.039930,0.010054,0.000137,0.006753,0.080075,-0.046682,-0.046633,-0.054844,0.055044
1,2000,Bragado,ampliacion,transicion,-35.000_-60.625,19.59,62000,55130,344.97,16.314426,...,0.063346,0.023782,-0.008197,-0.014946,-0.004877,0.068930,-0.078750,-0.073492,-0.047739,0.056858
2,2000,Carlos Tejedor,ampliacion,oeste,-35.500_-62.500,22.82,35100,33000,289.87,16.443607,...,0.043496,0.014228,-0.003047,-0.011262,-0.000597,0.027045,-0.068890,-0.007885,-0.054153,0.067002
3,2000,Chacabuco,ampliacion,transicion,-34.500_-60.625,24.44,79000,75500,365.65,16.672131,...,0.074623,0.024460,-0.000251,-0.002225,0.001607,0.090796,-0.105194,-0.040120,-0.019779,0.015967
4,2000,Colón,ampliacion,nucleo,-34.000_-61.250,31.74,53500,53500,385.74,17.042131,...,0.072776,0.010785,-0.016784,-0.018705,-0.008547,0.073267,-0.133939,-0.069471,-0.024488,0.045440
